In [56]:
# Import libraries

import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression

from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

In [57]:
# Load Bengaluru house dataset

df = pd.read_csv("Bengaluru_House_Data.csv")

df.head()

,location,size,total_sqft,bath,price
0,Electronic City Phase II,2 BHK,1056,2.0,39.07
1,Chikka Tirupathi,4 Bedroom,2600,5.0,120.00
2,Uttarahalli,3 BHK,1440,2.0,62.00
3,Lingadheeranahalli,3 BHK,1521,3.0,95.00
4,Kothanur,2 BHK,1200,2.0,51.00


In [58]:
# Shape

print(df.shape)


# Columns

print(df.columns)

(13320, 5)
Index(['location', 'size', 'total_sqft', 'bath', 'price'], dtype='object')


In [59]:
# Check missing values

df.isnull().sum()

location       1
size          16
total_sqft     0
bath          73
price          0
dtype: int64

In [60]:
# Check duplicates

print(df.duplicated().sum())


# Remove duplicates

df = df.drop_duplicates()


print(df.shape)

882
(12438, 5)


In [61]:
# Convert sqft range into average

def convert_sqft(x):

    try:

        if '-' in str(x):

            a,b = x.split('-')

            return (float(a)+float(b))/2

        return float(x)

    except:

        return np.nan


df['total_sqft'] = df['total_sqft'].apply(convert_sqft)

In [62]:
# Convert 2 BHK / 3 Bedroom

df['bhk'] = df['size'].str.extract('(\d+)')

df['bhk'] = pd.to_numeric(
    df['bhk'],
    errors='coerce'
)

In [63]:
# Fill bathroom missing values

df['bath'] = df['bath'].fillna(
    df['bath'].median()
)


# Remove remaining null rows

df = df.dropna()


df.isnull().sum()

location      0
size          0
total_sqft    0
bath          0
price         0
bhk           0
dtype: int64

In [64]:
# Price per sqft in rupees

df['price_per_sqft'] = (
    df['price'] * 100000
) / df['total_sqft']


df.head()

,location,size,total_sqft,bath,price,bhk,price_per_sqft
0,Electronic City Phase II,2 BHK,1056.0,2.0,39.07,2.0,3699.810606
1,Chikka Tirupathi,4 Bedroom,2600.0,5.0,120.00,4.0,4615.384615
2,Uttarahalli,3 BHK,1440.0,2.0,62.00,3.0,4305.555556
3,Lingadheeranahalli,3 BHK,1521.0,3.0,95.00,3.0,6245.890861
4,Kothanur,2 BHK,1200.0,2.0,51.00,2.0,4250.000000


In [65]:
# Remove unrealistic houses

df = df[
    df.total_sqft / df.bhk >= 300
]


# Remove extreme price values

df = df[
    (df.price_per_sqft >= df.price_per_sqft.quantile(0.05))
    &
    (df.price_per_sqft <= df.price_per_sqft.quantile(0.90))
]


print(df.shape)

(9903, 7)


In [66]:
# Keep location data for searching later

search_df = df.copy()

In [67]:
# Count houses in each location

location_count = df['location'].value_counts()


df['location_count'] = df['location'].map(
    location_count
)

In [68]:
# Keep locations with enough data

df = df[
    df.location_count >= 10
]

In [69]:
# Location ranking

location_details = df.groupby('location').agg(

    houses=('location','count'),

    avg_sqft_price=('price_per_sqft','mean'),

    avg_price_lakhs=('price','mean')

).reset_index()


location_details = location_details.sort_values(
    'avg_sqft_price'
)


location_details.head(10)

,location,houses,avg_sqft_price,avg_price_lakhs
119,Kereguddadahalli,10,3442.655403,34.250000
26,Banashankari Stage V,10,3705.071915,49.418000
124,Kothannur,19,3943.757796,49.702632
56,Doddakallasandra,11,3951.162985,53.530000
176,Sompura,11,3985.520523,48.272727
48,Channasandra,33,4021.518232,57.416970
197,Yelenahalli,12,4026.305111,51.185000
35,Begur Road,66,4100.666288,59.402879
43,Bommasandra,31,4118.472598,46.385806
41,Bisuvanahalli,35,4138.022142,42.685143


In [70]:
# Convert location text into numbers

df = pd.get_dummies(
    df,
    columns=['location'],
    drop_first=True
)

In [71]:
# Input features

X = df.drop(
    [
        'price',
        'size',
        'price_per_sqft'
    ],
    axis=1
)


# Output

y = df['price']

In [72]:
# Train test split

X_train,X_test,y_train,y_test = train_test_split(

    X,
    y,

    test_size=0.10,

    random_state=42
)

In [73]:
# Linear Regression

model = LinearRegression()


model.fit(
    X_train,
    y_train
)

,fit_intercept,True
,copy_X,True
,tol,1e-06
,n_jobs,None
,positive,False


In [74]:
# Prediction

y_pred = model.predict(X_test)


# Accuracy

accuracy = r2_score(
    y_test,
    y_pred
)


print(
    "Accuracy:",
    accuracy*100,
    "%"
)

Accuracy: 81.36436628295284 %


In [75]:
# Model errors

mae = mean_absolute_error(
    y_test,
    y_pred
)


rmse = np.sqrt(
    mean_squared_error(
        y_test,
        y_pred
    )
)


print("MAE:",mae)

print("RMSE:",rmse)

MAE: 15.640980086660335
RMSE: 22.824414380408783


In [77]:
# Location details

loc = input("Enter location: ")


result = search_df[
    search_df.location.str.lower()
    ==
    loc.lower()
]


print(result[
[
'location',
'total_sqft',
'bhk',
'bath',
'price',
'price_per_sqft'
]
].head(10))

       location  total_sqft  bhk  bath   price  price_per_sqft
5    Whitefield      1170.0  2.0   2.0   38.00     3247.863248
10   Whitefield      1800.0  3.0   2.0   70.00     3888.888889
27   Whitefield      1610.0  3.0   3.0   81.00     5031.055901
47   Whitefield      1459.0  2.0   2.0   94.82     6498.971899
52   Whitefield      2010.0  3.0   3.0   91.00     4527.363184
112  Whitefield      1116.0  2.0   2.0   51.91     4651.433692
163  Whitefield      4200.0  4.0   4.0  420.00    10000.000000
202  Whitefield      1225.0  2.0   2.0   47.60     3885.714286
203  Whitefield      1075.0  2.0   2.0   53.00     4930.232558
317  Whitefield      1280.0  2.0   2.0   75.00     5859.375000


In [78]:
# Search by sqft range

min_sqft = int(input("Minimum sqft: "))

max_sqft = int(input("Maximum sqft: "))


range_data = search_df[
    (search_df.total_sqft >= min_sqft)
    &
    (search_df.total_sqft <= max_sqft)
]


recommend = range_data.groupby('location').agg(

houses=('location','count'),

price_sqft=('price_per_sqft','mean'),

price_lakhs=('price','mean'),

bhk=('bhk','mean'),

bath=('bath','mean')

).reset_index()


recommend = recommend.sort_values(
    'price_sqft'
)


recommend.head(10)

,location,houses,price_sqft,price_lakhs,bhk,bath
296,K G Colony,1,3125.000000,50.0000,3.0,2.0
551,Ullal Uppanagar,1,3125.000000,75.0000,6.0,5.0
85,Banashankari Stage V,4,3150.235226,49.8525,3.0,3.0
169,Devara Jeevanahalli,1,3151.515152,52.0000,3.0,3.0
30,Abbigere,1,3240.000000,81.0000,6.0,6.0
128,Byrasandra,1,3254.437870,55.0000,3.0,2.0
402,Medahalli,1,3286.082474,51.0000,2.0,2.0
205,"Electronic city Phase 1,",1,3314.285714,58.0000,3.0,3.0
491,Samethanahalli,1,3333.333333,70.0000,3.0,3.0
203,"Electronic City Phase 1,",1,3333.333333,50.0000,3.0,3.0


In [79]:
# Top location analysis

top_locations = search_df.groupby('location').agg(

    total_houses=('location','count'),

    avg_price_sqft=('price_per_sqft','mean'),

    avg_price_lakhs=('price','mean')

).reset_index()


# Sort by lowest price per sqft

top_locations = top_locations.sort_values(
    'avg_price_sqft',
    ascending=True
)


print("TOP 10 BEST AFFORDABLE LOCATIONS")
print("--------------------------------")

print(
    top_locations.head(10)
)

TOP 10 BEST AFFORDABLE LOCATIONS
--------------------------------
                     location  total_houses  avg_price_sqft  avg_price_lakhs
501                K G Colony             1     3125.000000            50.00
291       Devara Jeevanahalli             1     3151.515152            52.00
674       Maruthi HBCS Layout             1     3160.000000            39.50
115         Asthagrama Layout             1     3166.666667            38.00
952      Thirumalashettyhally             1     3199.628598            34.46
871             Shakthi Nagar             1     3200.000000            36.00
566   Kenchanehalli R R Nagar             1     3200.000000            45.12
799     Raja Rajashweri Nagar             1     3200.000000            45.12
807      Rajarajesheari nagar             1     3200.000000            44.80
1034           Weavers Colony             1     3201.970443            26.00


In [80]:
# Final best location

best = top_locations.iloc[0]


print("BEST LOCATION")
print("-------------")

print(
    "Location:",
    best['location']
)

print(
    "Price per sqft ₹:",
    best['avg_price_sqft']
)

print(
    "Average Price Lakhs:",
    best['avg_price_lakhs']
)

print(
    "Available Houses:",
    best['total_houses']
)

BEST LOCATION
-------------
Location: K G Colony
Price per sqft ₹: 3125.0
Average Price Lakhs: 50.0
Available Houses: 1
